In [1]:
# Installa el cliente de InfluxDB y polarspara Python junto a paquetes necesarios
%pip install influxdb-client
%pip install polars

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Crear bucket
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS

URL = "http://localhost:8086"
TOKEN = "tokensomorrostro123456789"
ORG = "orgsomorrostro"
BUCKET = "Copia de seguridad"

with InfluxDBClient(url=URL, token=TOKEN, org=ORG) as client:
    buckets_api = client.buckets_api()

    # Buscar el bucket y crearlo únicamente si no existe
    bucket = buckets_api.find_bucket_by_name(BUCKET)
    if bucket is None:
        buckets_api.create_bucket(bucket_name=BUCKET, org=ORG)
        print(f"Bucket creado: {BUCKET}")
    else:
        print(f"El bucket ya existe: {BUCKET}")

    # Ejemplo de escritura; sustituye este punto por tus datos
    punto = Point("medicion").field("valor", 23.5)

    with client.write_api(write_options=SYNCHRONOUS) as write_api:
        write_api.write(bucket=BUCKET, org=ORG, record=punto)

ApiException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'Content-Type': 'application/json; charset=utf-8', 'X-Influxdb-Build': 'OSS', 'X-Influxdb-Version': 'v2.9.1', 'X-Platform-Error-Code': 'unauthorized', 'Date': 'Thu, 15 Oct 2026 11:23:39 GMT', 'Content-Length': '55'})
HTTP response body: {"code":"unauthorized","message":"unauthorized access"}


In [2]:
# Importar polars como pl y
import polars as pl

In [3]:
# Importa la librería de InfluxDB para Python
import influxdb_client

In [4]:
# Crea un cliente de InfluxDB con la URL, token y organización especificados, si sale true significa que la conexión fue exitosa, si sale false significa que hubo un error en la conexión
from influxdb_client import InfluxDBClient

client = InfluxDBClient(
    url="http://localhost:8086",
    token="tokensomorrostro123456789",
    org="orgsomorrostro"
)

print(client.ping())

True


In [5]:
# Lee el archivo CSV "steam_data.csv" y lo carga en un DataFrame de Polars
df = pl.read_csv("steam_data.csv")

In [6]:
df = pl.read_csv(
    "steam_data.csv",
    separator=";",
    try_parse_dates=True
)

df_juegos = df.filter(
    pl.col("Game").str.contains("(?i)counter.?strike|among us")
)

print(df_juegos.select("Game").unique())
print(df_juegos.shape)
print(df_juegos.head(2))

shape: (2, 1)
┌──────────────────┐
│ Game             │
│ ---              │
│ str              │
╞══════════════════╡
│ Counter-Strike 2 │
│ Among Us         │
└──────────────────┘
(11384, 4)
shape: (2, 4)
┌──────────┬─────────────────────┬─────────┬─────────────────┐
│ Game     ┆ DateTime            ┆ Players ┆ Average Players │
│ ---      ┆ ---                 ┆ ---     ┆ ---             │
│ str      ┆ datetime[μs]        ┆ i64     ┆ str             │
╞══════════╪═════════════════════╪═════════╪═════════════════╡
│ Among Us ┆ 2018-11-16 00:00:00 ┆ 8       ┆                 │
│ Among Us ┆ 2018-11-17 00:00:00 ┆ 17      ┆                 │
└──────────┴─────────────────────┴─────────┴─────────────────┘


In [7]:
# Convierte la columna "Average Players" a tipo float64, eliminando las comas y reemplazándolas por nada
df_envio = df_juegos.with_columns(
    pl.col("Average Players")
      .str.replace_all(",", "")
      .cast(pl.Float64, strict=False)
)

In [ ]:
# Subir los datos a InfluxDB
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

client = InfluxDBClient(
    url="http://localhost:8086",
    token="tokensomorrostro123456789",
    org="orgsomorrostro",
    timeout=60000
)

write_api = client.write_api(write_options=SYNCHRONOUS)

# Crear los puntos a partir del DataFrame filtrado de Polars
puntos = []

for fila in df_envio.iter_rows(named=True):
    punto = (
        Point("steam_data")
        .tag("game", fila["Game"])
        .time(fila["DateTime"], WritePrecision.US)
    )

    if fila["Players"] is not None:
        punto = punto.field("players", int(fila["Players"]))

    if fila["Average Players"] is not None:
        punto = punto.field("average_players", float(fila["Average Players"]))

    puntos.append(punto)

# Subirlos en lotes de 500
bucket = "Copia seguridad"
lote_tamano = 500

for inicio in range(0, len(puntos), lote_tamano):
    lote = puntos[inicio:inicio + lote_tamano]
    write_api.write(bucket=bucket, org="orgsomorrostro", record=lote)
    print(f"Enviados {min(inicio + lote_tamano, len(puntos))} de {len(puntos)}")

client.close()

Enviados 500 de 11384
Enviados 1000 de 11384
Enviados 1500 de 11384
Enviados 2000 de 11384
Enviados 2500 de 11384
Enviados 3000 de 11384
Enviados 3500 de 11384
Enviados 4000 de 11384
Enviados 4500 de 11384
Enviados 5000 de 11384
Enviados 5500 de 11384
Enviados 6000 de 11384
Enviados 6500 de 11384
Enviados 7000 de 11384
Enviados 7500 de 11384
Enviados 8000 de 11384
Enviados 8500 de 11384
Enviados 9000 de 11384
Enviados 9500 de 11384
Enviados 10000 de 11384
Enviados 10500 de 11384
Enviados 11000 de 11384
Enviados 11384 de 11384
